## Objective

Investigate **semantic basins in latent space** using real sentence embeddings from
`sentence-transformers/all-MiniLM-L6-v2` (384-D, L2-normalised). The corpus is the same
96-sentence / 8-field dataset used in Notebook 03 so results are directly comparable.

Three comparison conditions:

1. **Vanilla algorithms alone** — KDE, Diffusion Maps, Basin-Hopping on PCA-2D projections.
2. **Vanilla + R_spec** — spectral augmentation via `aspace.search(alpha=0.05)` blended with
   each vanilla score (README Principle 3: only the spectral component may augment vanilla).
3. **ArrowSpace λ80 alone** — `aspace.search(alpha=0.80)` as a direct competitive baseline.

All λ-scores come from `aspace.search(...)` — no manual Laplacian reconstruction (README Principle 0).

> **Magnification note**: ArrowSpace works best with amplified embedding magnitude.
> `X_arrow = X_base * ARROW_MAG` (with `ARROW_MAG = 1.12`) is used **only** for the
> ArrowSpace build and search steps. All vanilla branches (PCA, KDE, DiffMaps, BasinHop,
> cosine metrics) use the original L2-normalised `X_base` so the comparison remains clean.


In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import json, os, warnings
warnings.filterwarnings('ignore')

os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

from sklearn.metrics.pairwise import rbf_kernel
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from scipy.stats import gaussian_kde
from scipy.linalg import eigh
from scipy.optimize import basinhopping

from arrowspace import ArrowSpaceBuilder

os.makedirs('output__02', exist_ok=True)
COLORS8 = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B3','#937860','#DA8BC3','#64B5CD']
RNG = np.random.default_rng(42)

# ── Magnification factor ────────────────────────────────────────────────────
# ArrowSpace works best with amplified embedding magnitude. X_arrow is used
# ONLY for ArrowSpace build/search; vanilla branches always use X_base.
ARROW_MAG = 1.0

## 1. Sentence corpus — same 96-sentence / 8-field dataset as Notebook 03

12 sentences per field, encoded with `all-MiniLM-L6-v2` (384-D). An offline synthetic fallback
is provided if the model cannot be loaded.

A held-out boundary set of ambiguous / bridge-like sentences is kept for downstream probing
(same as NB03) but not used to define fields.


In [2]:
FIELD_SENTENCES = {
    "astronomy": [
        "The telescope tracked a faint comet across the winter sky.",
        "Astronomers estimated the planet's orbit from repeated measurements.",
        "A nebula glowed behind the dense field of stars.",
        "The satellite adjusted its path after the eclipse window closed.",
        "Researchers measured parallax to estimate the star's distance.",
        "The observatory logged a burst of radiation from a pulsar.",
        "A meteor shower peaked just before dawn over the valley.",
        "The spacecraft photographed a cratered moon near the gas giant.",
        "Gravity bent the path of light around the massive object.",
        "The quasar appeared bright even at extreme distance.",
        "The constellation was visible above the horizon after sunset.",
        "The probe transmitted data from the edge of the magnetosphere."
    ],
    "programming": [
        "The compiler rejected the function because the types did not match.",
        "She refactored the module to reduce duplicated logic.",
        "A race condition appeared when two threads touched the same state.",
        "The debugger stopped at the line that mutated the array.",
        "The runtime cached the result to avoid repeated computation.",
        "A recursive function walked the tree until it found a leaf.",
        "The engineer pushed a patch after the failing test reproduced locally.",
        "The parser consumed tokens until it reached the closing brace.",
        "A pointer bug corrupted memory during the benchmark.",
        "The iterator yielded one record at a time from the stream.",
        "The service exposed a clean API for the client application.",
        "The deployment pipeline rolled back after the health check failed."
    ],
    "cooking": [
        "The chef simmered the broth until the flavours deepened.",
        "She whisked the sauce while the butter slowly melted.",
        "The dough rested before it went into the hot oven.",
        "A pinch of zest brightened the rich stew.",
        "They braised the vegetables until they turned soft and glossy.",
        "The pan was hot enough to sear the meat quickly.",
        "He julienned the carrots into thin even strips.",
        "The cook deglazed the skillet with white wine.",
        "Fresh herbs lifted the aroma of the roasted dish.",
        "The pastry needed another minute before the crust browned.",
        "She kneaded the dough until it felt smooth and elastic.",
        "The stockpot filled the kitchen with a savoury smell."
    ],
    "finance": [
        "The fund increased its hedge after market volatility returned.",
        "Rising inflation reduced the real yield on the bond.",
        "The analyst updated the valuation after the earnings call.",
        "The portfolio shifted toward higher-liquidity assets.",
        "A dividend increase signalled confidence from the board.",
        "The bank tightened collateral rules for new loans.",
        "Investors watched the maturity profile of the debt closely.",
        "The ledger showed a premium paid for the acquisition.",
        "The trader closed the arbitrage spread before the market moved.",
        "Cash flow improved after the company refinanced its liabilities.",
        "The repo market reflected short-term funding stress.",
        "A coupon payment arrived at the end of the quarter."
    ],
    "emotions": [
        "She felt a sudden wave of joy when the letter arrived.",
        "A quiet sense of dread settled over the empty room.",
        "His apology eased some of her lingering resentment.",
        "The reunion filled them with nostalgia and warmth.",
        "Anxiety made the wait feel much longer than it was.",
        "The painting inspired awe in almost every visitor.",
        "He spoke with affection even after the argument.",
        "Grief returned sharply at the sound of the old song.",
        "Their success brought relief more than excitement.",
        "Envy faded once she understood the work behind the result.",
        "The child looked at the stage with wonder.",
        "Contentment replaced the earlier tension by evening."
    ],
    "anatomy": [
        "The tendon connects muscle to bone at the joint.",
        "Signals travelled along the neuron toward the spinal cord.",
        "The surgeon examined the ventricle on the scan.",
        "Cartilage protected the knee from constant friction.",
        "The retina converts light into neural signals.",
        "Blood left the heart through the aorta.",
        "The cortex supports several higher cognitive functions.",
        "The ligament stabilised the ankle after the twist.",
        "The cornea refracts incoming light before it reaches the lens.",
        "Marrow inside the femur produces blood cells.",
        "The larynx controls airflow and helps generate speech.",
        "A synapse transmits information between neighbouring neurons."
    ],
    "music": [
        "The melody returned in a softer register near the end.",
        "A suspended chord delayed the harmonic resolution.",
        "The orchestra followed the conductor through the crescendo.",
        "Syncopation gave the rhythm a restless energy.",
        "The pianist shaped the phrase with delicate rubato.",
        "A low drone supported the vocal line underneath.",
        "The cadence landed cleanly in the final bar.",
        "Her vibrato widened during the sustained note.",
        "The fugue introduced each voice in careful sequence.",
        "Timbre mattered more than volume in the recording.",
        "The sonata opened with a tense repeated motif.",
        "A sudden diminuendo changed the emotional colour of the passage."
    ],
    "geography": [
        "The glacier carved a broad valley through the mountain range.",
        "Seasonal monsoon winds reshaped the coastal shoreline.",
        "The river widened into an estuary near the sea.",
        "A narrow isthmus joined the two larger landmasses.",
        "The plateau rose above the surrounding plain.",
        "Satellite maps traced the watershed across the region.",
        "The peninsula extended into the cold northern gulf.",
        "Heavy erosion deepened the canyon over time.",
        "The archipelago sits far beyond the main shipping route.",
        "Latitude strongly affects daylight in the winter months.",
        "The fjord cut inland between steep rocky slopes.",
        "Topography shaped settlement patterns across the basin."
    ]
}

BOUNDARY_SENTENCES = [
    "The bank of monitors showed a current of live market data.",
    "The bridge section modulated before the final chorus returned.",
    "A root process spawned another thread after the system reboot.",
    "The pitch of the proposal changed after the investor meeting.",
    "Mercury moved quickly across the morning sky above the harbour.",
    "The delta model shifted after new river measurements arrived.",
    "The cell line was stored beside the culture medium in the lab.",
    "Spring light changed the colour of the valley by noon.",
    "The key passage unlocked the argument in the final chapter.",
    "The scale of the map made the ridge look much smaller."
]

sentences, labels = [], []
for field, sents in FIELD_SENTENCES.items():
    for s in sents:
        sentences.append(s)
        labels.append(field)

LABELS = np.array(labels)
FIELD_NAMES = list(FIELD_SENTENCES.keys())
N_ITEMS = len(sentences)
print(f'Corpus: {N_ITEMS} sentences across {len(FIELD_NAMES)} semantic fields')
print('Items per field:', {k: len(v) for k, v in FIELD_SENTENCES.items()})
print(f'Held-out boundary sentences: {len(BOUNDARY_SENTENCES)}')

Corpus: 96 sentences across 8 semantic fields
Items per field: {'astronomy': 12, 'programming': 12, 'cooking': 12, 'finance': 12, 'emotions': 12, 'anatomy': 12, 'music': 12, 'geography': 12}
Held-out boundary sentences: 10


## 2. Encode with all-MiniLM-L6-v2

L2-normalised 384-D embeddings. The loader matches NB03: `load_latent_space` accepts the
main corpus and an optional `extra_texts` list (used here for `BOUNDARY_SENTENCES`).
An offline fallback synthesises the same structure if the model is unavailable
(Gaussian clusters + small within-field noise, seeded via `RNG`).


In [3]:
def l2norm(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)


def load_latent_space(texts, extra_texts=None):
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
        X = model.encode(texts, normalize_embeddings=True, show_progress_bar=False).astype(np.float64)
        P = None
        if extra_texts:
            P = model.encode(extra_texts, normalize_embeddings=True, show_progress_bar=False).astype(np.float64)
        return X, P, "all-MiniLM-L6-v2 (sentence-transformers)", model
    except Exception as e:
        print(f'Real encoder unavailable ({e!r}) — synthesising sentence corpus latent space.')
        F = 384
        n_fields = len(FIELD_NAMES)
        centers = RNG.normal(size=(n_fields, F))
        X = []
        for lab in LABELS:
            c = centers[FIELD_NAMES.index(lab)]
            X.append(c + 0.45 * RNG.normal(size=F))
        X = l2norm(np.vstack(X).astype(np.float64))
        P = None
        if extra_texts:
            P = []
            for _ in extra_texts:
                i, j = RNG.integers(0, n_fields, size=2)
                vec = centers[i] + centers[j] + 0.6 * RNG.normal(size=F)
                P.append(vec)
            P = l2norm(np.vstack(P).astype(np.float64))
        return X, P, "synthetic-384D", None


X_high, P_boundary, SOURCE, MODEL = load_latent_space(sentences, BOUNDARY_SENTENCES)

# ── Split into vanilla base and ArrowSpace-only magnified copy ──────────────
# X_base  — original L2-normalised embeddings used by ALL vanilla branches
#            (PCA, KDE, DiffMaps, BasinHop, cosine metrics)
# X_arrow — X_base * ARROW_MAG used ONLY for ArrowSpace build and search
X_base  = np.ascontiguousarray(X_high, dtype=np.float64)
X_arrow = np.ascontiguousarray(X_base * ARROW_MAG, dtype=np.float64)

# Aliases kept for compatibility with downstream cells
labels = LABELS
N = N_ITEMS
rng = RNG
N, F = X_base.shape
print(f'Embeddings: N={N}  F={F}  source={SOURCE}')
print(f'X_base norms (mean): {np.linalg.norm(X_base, axis=1).mean():.4f}')
print(f'X_arrow norms (mean): {np.linalg.norm(X_arrow, axis=1).mean():.4f}  (ARROW_MAG={ARROW_MAG})')
if P_boundary is not None:
    print(f'Boundary set: {P_boundary.shape[0]} sentences, {P_boundary.shape[1]} dims')

# PCA-2D for visualisation and KDE / DiffMaps surface — always on X_base
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_base)
x2, y2 = X_2d[:, 0], X_2d[:, 1]
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%  '
      f'(PC1 {pca.explained_variance_ratio_[0]*100:.1f}%, PC2 {pca.explained_variance_ratio_[1]*100:.1f}%)')

Embeddings: N=96  F=384  source=all-MiniLM-L6-v2 (sentence-transformers)
X_base norms (mean): 1.0000
X_arrow norms (mean): 1.0000  (ARROW_MAG=1.0)
Boundary set: 10 sentences, 384 dims
PCA explained variance: 11.2%  (PC1 6.3%, PC2 4.9%)


## 3. ArrowSpace index — R_spec and λ80 via `aspace.search()`

- `alpha=0.05` → spectral-dominant `R_spec` — augmentation term.
- `alpha=0.80` → balanced `λ80` — standard ArrowSpace search, evaluated independently.

Graph params match NB03 (eps=2.0, k=25, unit-sphere geometry).

> **ArrowSpace is built on `X_arrow` (magnified by `ARROW_MAG=1.12`). Queries also use
> `X_arrow`. Vanilla branches are never touched by this scaling.**


In [ ]:
GRAPH_PARAMS = {'eps': 1.9, 'k': 25, 'topk': 10, 'p': 2.0, 'sigma': 0.5}

# Build ArrowSpace on the magnified space only
aspace, gl = (
        ArrowSpaceBuilder()
        .with_seed(42)
        .with_dims_reduction(enabled=False, eps=None)
        .with_sampling("simple", 1.0)
    ).build_and_store(GRAPH_PARAMS, X_arrow)
print(f'ArrowSpace index built on X_arrow (ARROW_MAG={ARROW_MAG}).')
print('Sorted λ (first 10):', aspace.lambdas_sorted()[:10])


def extract_scores(aspace, gl, X, alpha, on_fail="high"):
    X = np.ascontiguousarray(X, dtype=np.float64)
    raw = np.zeros(len(X), dtype=np.float64)

    for i, x in enumerate(X):
        try:
            hits = aspace.search(x, gl, float(alpha))
        except ValueError as e:
            if "Lambda is zero" in str(e):
                print(f"vector at position {i} has 0.0 lambda, assigning a high lambda")
                print("sentence: ", sentences[i])
                raw[i] = 1.0 if on_fail == "high" else np.nan
                continue
            raise

        found = False
        for idx, score in hits:
            if idx == i:
                raw[i] = score
                found = True
                break
        if not found:
            raw[i] = min(s for _, s in hits) if len(hits) else (1.0 if on_fail == "high" else np.nan)
    return raw


# Query with X_arrow — same magnified space the index was built on
print('Extracting R_spec  (alpha=0.05) …')
R_spec_raw = extract_scores(aspace, gl, X_arrow, alpha=0.05)
R_spec = (R_spec_raw - R_spec_raw.min()) / (R_spec_raw.max() - R_spec_raw.min() + 1e-9)

print('Extracting λ80 (alpha=0.80) …')
L80_raw = extract_scores(aspace, gl, X_arrow, alpha=0.80)
L80 = (L80_raw - L80_raw.min()) / (L80_raw.max() - L80_raw.min() + 1e-9)

as_spec_is_min = R_spec <= np.quantile(R_spec, 0.10)
as_80_is_min   = L80   <= np.quantile(L80,    0.10)
print(f'R_spec minima: {as_spec_is_min.sum()}  |  λ80 minima: {as_80_is_min.sum()}')

ArrowSpace index built on X_arrow (ARROW_MAG=1.0).
Sorted λ (first 10): [(0.0, 61), (0.0025395127638494224, 1), (0.004345008518989705, 86), (0.0043806509133247785, 91), (0.004539074690065296, 17), (0.00595233460020482, 66), (0.007119440687138998, 65), (0.007220678950980648, 67), (0.007507390981814913, 28), (0.007520490200575004, 44)]
Extracting R_spec  (alpha=0.05) …
vector at position 61 has 0.0 lambda, assigning a high lambda
sentence:  Signals travelled along the neuron toward the spinal cord.
Extracting λ80 (alpha=0.80) …
vector at position 61 has 0.0 lambda, assigning a high lambda
sentence:  Signals travelled along the neuron toward the spinal cord.
R_spec minima: 96  |  λ80 minima: 16


## 4. Vanilla algorithms

KDE and Basin-Hopping operate on PCA-2D; Diffusion Maps operates on the full 384-D embeddings
(using an RBF kernel on L2-normalised vectors, same approach as NB03).

> All vanilla methods use **`X_base`** — the original, un-magnified embeddings.


In [5]:
# ── 4a  KDE on PCA-2D ───────────────────────────────────────────────
kde = gaussian_kde(X_2d.T, bw_method=0.30)
kde_density_norm = (lambda d: (d - d.min()) / (d.max() - d.min() + 1e-9))(kde(X_2d.T))
kde_vanilla_score = 1.0 - kde_density_norm
kde_is_min = kde_vanilla_score <= np.quantile(kde_vanilla_score, 0.10)

# ── 4b  Diffusion Maps on 384-D X_base ──────────────────────────────
dots = np.clip(X_base @ X_base.T, -1, 1)
angles = np.arccos(dots)
sigma2_diff = float(np.median(angles[angles > 0]) ** 2)
K_rbf = np.exp(-angles**2 / (2 * sigma2_diff))
P_diff = np.diag(1.0 / K_rbf.sum(axis=1)) @ K_rbf

eigvals, eigvecs = eigh(P_diff, subset_by_index=[N - 6, N - 1])
eigvals, eigvecs = eigvals[::-1], eigvecs[:, ::-1]

diff_coords = eigvecs[:, 1:3] * eigvals[np.newaxis, 1:3]
diff_dist_n = (lambda d: (d - d.min()) / (d.max() - d.min() + 1e-9))(
    np.linalg.norm(diff_coords - diff_coords.mean(0), axis=1))
diff_vanilla_score = diff_dist_n
diff_is_min = diff_vanilla_score <= np.quantile(diff_vanilla_score, 0.10)

# ── 4c  Basin-Hopping on KDE surface ─────────────────────────────────────
def neg_log_kde(pt):
    v = kde(np.array(pt).reshape(2, 1)).item()
    return -np.log(float(v) + 1e-20)

seeds = [X_2d.mean(0) + 0.4 * rng.standard_normal(2) for _ in range(14)]
bh_raw = [basinhopping(neg_log_kde, s,
              minimizer_kwargs={'method': 'Nelder-Mead',
                  'options': {'xatol': 1e-3, 'fatol': 1e-3, 'maxiter': 300}},
              niter=60, T=1.2, stepsize=0.4, seed=42).x for s in seeds]

agg = AgglomerativeClustering(n_clusters=None, distance_threshold=0.3, linkage='single')
agg.fit(np.array(bh_raw))
bh_minima = np.array([np.array(bh_raw)[agg.labels_ == c].mean(0)
                       for c in np.unique(agg.labels_)])
bh_dist_n = (lambda d: (d - d.min()) / (d.max() - d.min() + 1e-9))(
    np.array([np.linalg.norm(X_2d - m, axis=1) for m in bh_minima]).min(0))
bh_vanilla_score = bh_dist_n
bh_is_min = bh_vanilla_score <= np.quantile(bh_vanilla_score, 0.10)

print(f'Basin-Hopping: {len(bh_minima)} unique minima found')

Basin-Hopping: 1 unique minima found


## 5. Spectral augmentation — vanilla + R_spec

Formula: `aug(x) = α·vanilla(x) + (1−α)·R_spec(x)`.
Only `R_spec` (α=0.05) is the augmentation term; λ80 is evaluated as a standalone condition.


In [6]:
ALPHA = 0.50

kde_aug_score  = ALPHA * kde_vanilla_score  + (1 - ALPHA) * R_spec
diff_aug_score = ALPHA * diff_vanilla_score + (1 - ALPHA) * R_spec
bh_aug_score   = ALPHA * bh_vanilla_score   + (1 - ALPHA) * R_spec

kde_aug_is_min  = kde_aug_score  <= np.quantile(kde_aug_score,  0.10)
diff_aug_is_min = diff_aug_score <= np.quantile(diff_aug_score, 0.10)
bh_aug_is_min   = bh_aug_score   <= np.quantile(bh_aug_score,   0.10)

## 6. Quality metrics

In [7]:
def purity(mask, lab=labels):
    sel = lab[mask]
    vals, counts = np.unique(sel, return_counts=True)
    return float(counts.max() / len(sel)) if len(sel) else 0.0

def jaccard(a, b):
    return float((a & b).sum()) / float((a | b).sum() + 1e-9)

def mean_cosine(mask):
    # cosine similarity always computed on X_base (un-magnified)
    V = X_base[mask]
    if len(V) < 2:
        return 1.0
    sims = V @ V.T
    n = len(V)
    return float((sims.sum() - n) / (n * (n - 1)))

all_masks = [
    as_spec_is_min, as_80_is_min,
    kde_is_min, kde_aug_is_min,
    diff_is_min, diff_aug_is_min,
    bh_is_min, bh_aug_is_min,
]
all_names = [
    'ArrowSpace (\u03b1=0.05 spectral)', 'ArrowSpace (\u03b1=0.80 balanced)',
    'KDE (vanilla)', 'KDE + R_spec (aug)',
    'DiffMaps (vanilla)', 'DiffMaps + R_spec (aug)',
    'BasinHop (vanilla)', 'BasinHop + R_spec (aug)',
]

cmp_df = pd.DataFrame([{
    'Method': name,
    'Cluster purity': round(purity(mask), 3),
    'Jaccard w/ AS spectral': round(jaccard(mask, as_spec_is_min), 3),
    'Mean R_spec (norm)': round(float(R_spec[mask].mean()), 4),
    'Mean cosine sim': round(mean_cosine(mask), 4),
} for name, mask in zip(all_names, all_masks)])

cmp_df

,Method,Cluster purity,Jaccard w/ AS spectral,Mean R_spec (norm),Mean cosine sim
0,ArrowSpace (α=0.05 spectral),0.125,1.000,0.0,0.1023
1,ArrowSpace (α=0.80 balanced),0.250,0.167,0.0,0.1189
2,KDE (vanilla),0.500,0.104,0.0,0.1223
3,KDE + R_spec (aug),0.500,0.104,0.0,0.1223
4,DiffMaps (vanilla),0.500,0.104,0.0,0.1357
5,DiffMaps + R_spec (aug),0.500,0.104,0.0,0.1357
6,BasinHop (vanilla),0.600,0.104,0.0,0.1222
7,BasinHop + R_spec (aug),0.600,0.104,0.0,0.1222
